## XBRL US API - Python example  
This sample Python code queries the XBRL US Public Filings Database.
### Authenticate for access token 
Run the cell below, then type your XBRL US Web account email, account password, Client ID, and secret (get these from https://xbrl.us/access-token), pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
%pip install -q bs4
%pip install -q bleach
print('Enter your XBRL US Web account email: ')
from bs4 import BeautifulSoup
import bleach
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode
from pandas.core.base import textwrap
email = input()
password = getpass.getpass(prompt='Password: ')
clientid = getpass.getpass(prompt='Client ID: ')
secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : ''.join(email), 
            'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'password' : ''.join(password), 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

payload = urlencode(body_auth)
url = 'https://api.xbrl.us/oauth2/token'
headers = {"Content-Type": "application/x-www-form-urlencoded"}

res = requests.request("POST", url, data=payload, headers=headers)
auth_json = res.json()

if 'error' in auth_json:
    print ("\n\nThere was a problem generating an access token with these credentials. Run the first cell again to enter credentials.")
else:
    print ("\n\nYour access token expires in 60 minutes. After it expires, run the cell immediately below this one to generate a new token and continue to use the query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
access_token = auth_json['access_token']
refresh_token = auth_json['refresh_token']
newaccess = ''
newrefresh = ''
#print('access token: ' + access_token + ' refresh token: ' + refresh_token)

#### Refresh token 
The cell below is only needed to refresh an expired access token after 60 minutes. When the access token no longer returns results, run the cell below to refresh the access token or re-enter credentials by running the cell above. Until the refresh token process is needed, skip ahead to **Make a Query**. 


In [ ]:
token = token if newrefresh != '' else refresh_token 

refresh_auth = {'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'grant_type' : 'refresh_token', 
            'platform' : 'ipynb', 
            'refresh_token' : ''.join(token) }
refreshres = requests.post(url, data=refresh_auth)
refresh_json = refreshres.json()
access_token = refresh_json['access_token']
refresh_token = refresh_json['refresh_token']#print('access token: ' + access_token + 'refresh token: ' + refresh_token)
print('Your access token is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.')
print(access_token)

### Make a query 
After the access token confirmation appears above, you can modify the query below, then use the **_Cell >> Run_** menu option from the cell **immediately below this text** to run the entire query for results.

Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return. 

In [ ]:
# Define the parameters for the filter and fields to be returned, 
# run the loop to return results.
offset_value = 0
res_df = []

# This query strips HTML from non-monetary fact.values in a report. 
# The results include a fact.value-link (rendered fact.value) in the dataframe.

endpoint = "fact" #see https://xbrlus.github.io/xbrl-api for documentation on valid endpoints

var_id = [602860
           ]
# Define data fields to return (multi-sort based on order)

fields = [ # this is the list of the characteristics of the data being returned by the query
         'fact.value-link',
         'fact.value',
         'concept.local-name.sort(ASC)',
         ]

string_id = [str(int) for int in var_id]

params = { # this is the list of what's being queried against the search endpoint
         'report.id': ','.join(string_id),
         'concept.is-monetary': 'false',
         'fields': ','.join(fields)
         }

# DO NOT EDIT BELOW THIS LINE
search_endpoint = 'https://api.xbrl.us/api/v1/'+endpoint+'/search'

# Execute the query with loop for all results
orig_fields = params['fields']

count = 0
query_start = datetime.now()
printed = False
while True:
    if not printed:
        print("On", query_start.strftime("%c"), email, "(client ID:", str(clientid.split('-')[0]), "...) started the query and")
        printed = True
    res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(access_token)})
    res_json = res.json()
    if 'error' in res_json:
        print('There was an error: {}'.format(res_json['error_description']))
        break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',fact.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',fact.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',fact.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
        
    # results as dataframe
    df = pd.DataFrame(res_df)

    # convert fact.value column to string
    dfact = df['fact.value'].astype("string")
    #print(dfact)


On Mon Sep  4 20:28:40 2023 xxx@xbrl.us (client ID: 69e1257c ...) started the query and
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 469 records.

At Mon Sep  4 20:28:41 2023, the query finished with   469   rows returned in 0:00:00.815076 for 
https://api.xbrl.us/api/v1/fact/search?report.id=602860&concept.is-monetary=false&fields=fact.value-link,fact.value,concept.local-name.sort(ASC)


In [ ]:
    # # ORIGINAL VIEW - the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    # pd.options.display.float_format = '{:,.2f}'.format
    # display(HTML(df.to_html()))    

    # # OPTION 1. REMOVE TAGS - this makes table data in text blocks unreadable.
    # def remove_html_tags(df):
    #  soup = BeautifulSoup(df, 'html.parser')
    #  return soup.get_text()   
    
    # df['fact.value'] = dfact.apply(remove_html_tags)

    # # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    # pd.options.display.float_format = '{:,.2f}'.format
    # display(HTML(df.to_html()))    

    # # OPTION 2. REMOVE DEFINED ATTRIBUTES - this works for html sample but not df or dfact output - ?
    # # HTML sample for testing
    # html="""<table width="100"><tr style="display:inline"><td colspan="1" rowspan="1" valign="top">size</td></tr></table>"""
    
    # soup = BeautifulSoup(html, 'html.parser') # this cleans the html sample above; replace html with df on this line and it does not work.

    # REMOVE_ATTRIBUTES = [
    # 'lang','language','onmouseover','onmouseout','script','style','font',
    # 'dir','face','size','color','style','class','width','height','hspace',
    # 'border','valign','align','background','bgcolor','text','link','vlink',
    # 'alink','cellpadding','cellspacing','colspan','rowspan']

    # def CleanSoup(content):
    #     #if soup.str.contains('<'): # for each dfact that contains < and >
    #         for attribute in REMOVE_ATTRIBUTES:
    #             for tag in soup.find_all(attrs={attribute: True}):
    #                 del tag[attribute]
    #         return content
    # print(html)
    # print(CleanSoup(soup))
    
    # OPTION 3. use bleach to display HTML in Python output https://gist.github.com/connerxyz/46d7cb22392705b478eb4cd782952c7e
    # NOTE: results are still exported with inline HTML styles
    
    def bleached_df_table(df, class_map={}):
        result = df.to_html(escape=False)
        result = bleach.clean(
            result,
            tags=['table', 'thead', 'tbody', 'tr', 'th', 'td', 'h1', 'h2', 'h3', 'h4', 'p', 'b', 'i', 'em', 'u'], 
            strip=True,
        )
        return result
    display(HTML(bleached_df_table(df)))

,fact.value-link,fact.value,concept.local-name
0,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464929,P0Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife
1,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464829,P9Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife
2,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464852,P6Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife
3,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464931,P9Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife
4,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464927,P4Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife
5,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464935,P4Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife
6,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464943,P7Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife
7,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464933,P5Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife
8,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464925,P7Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife
9,https://csuite.xbrl.us/php/dispatch.php?Task=htmlExportFact&FactID=394464937,P3Y,AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife


In [ ]:
# If you run this program locally, you can save the output to a file on your computer (modify D:\results.csv to your system)
df.to_csv(r"D:\results.csv",sep=",")